# S3 Data Intelligence with Snowflake Cortex AI

## Build an AI-powered document intelligence pipeline in 90 minutes

**What you'll build:**
- Auto-ingest PDFs from S3 into Snowflake via Snowpipe
- Process them with Cortex AI functions (parse, extract, classify, summarize, redact, embed)
- Search across all content with Cortex Search
- Query structured data with natural language via Cortex Analyst
- Chat with a unified Cortex Agent via Snowflake Intelligence
- Build a Streamlit chat frontend

**Architecture:**
```
S3 (PDFs) → Snowpipe → AI Functions → Intelligence Table
                                            ↓
                            Cortex Search + Cortex Analyst
                                            ↓
                                      Cortex Agent
                                            ↓
                         Snowflake Intelligence / Streamlit App
```

---
### Before you start:
1. You need an S3 bucket with prefix `healthcare/pdfs/` containing the sample PDF files
2. An IAM role for Snowflake access (see LAB_GUIDE.md for AWS setup)

---
## Step 1: Create Database & Warehouse (2 min)

We create a simple structure: RAW for file metadata, PROCESSED for AI-enriched data, ANALYTICS for structured data and the agent.

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE DATABASE HEALTHCARE_AI_DEMO;
CREATE SCHEMA IF NOT EXISTS HEALTHCARE_AI_DEMO.RAW;
CREATE SCHEMA IF NOT EXISTS HEALTHCARE_AI_DEMO.PROCESSED;
CREATE SCHEMA IF NOT EXISTS HEALTHCARE_AI_DEMO.ANALYTICS;

CREATE OR REPLACE WAREHOUSE HEALTHCARE_AI_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE;

USE WAREHOUSE HEALTHCARE_AI_WH;

---
## Step 2: Connect to S3 (5 min)

A storage integration establishes trust between Snowflake and AWS — no credentials stored in SQL.

**Replace** `<YOUR_BUCKET_NAME>` and `<YOUR_AWS_IAM_ROLE_ARN>` with your values.

> **Important:** Do NOT re-run this cell after updating the IAM trust policy. It generates a new external ID each time.

In [ ]:
CREATE STORAGE INTEGRATION IF NOT EXISTS HEALTHCARE_S3_INTEGRATION
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'S3'
  ENABLED = TRUE
  STORAGE_AWS_ROLE_ARN = '<YOUR_AWS_IAM_ROLE_ARN>'
  STORAGE_ALLOWED_LOCATIONS = ('s3://<YOUR_BUCKET_NAME>/healthcare/');

### Get Snowflake's IAM User ARN & External ID

From the output below, copy **`STORAGE_AWS_IAM_USER_ARN`** and **`STORAGE_AWS_EXTERNAL_ID`**.

Then update your IAM role trust policy in AWS:

1. Go to **AWS Console → IAM → Roles → your role → Trust relationships → Edit trust policy**
2. Replace the trust policy with:

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal": {
        "AWS": "<STORAGE_AWS_IAM_USER_ARN from below>"
      },
      "Action": "sts:AssumeRole",
      "Condition": {
        "StringEquals": {
          "sts:ExternalId": "<STORAGE_AWS_EXTERNAL_ID from below>"
        }
      }
    }
  ]
}
```

3. **Save** and wait **30 seconds** before running the next cell.

> ⚠️ If you re-run the storage integration cell above, a **new** External ID is generated and you must update the trust policy again.

In [ ]:
%%sql -r dataframe_3
DESCRIBE INTEGRATION HEALTHCARE_S3_INTEGRATION;

### Create External Stage

In [ ]:
CREATE OR REPLACE STAGE HEALTHCARE_AI_DEMO.RAW.S3_MEDICAL_DOCS
  URL = 's3://<YOUR_BUCKET_NAME>/healthcare/pdfs/'
  STORAGE_INTEGRATION = HEALTHCARE_S3_INTEGRATION
  DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE);

### Verify — should list your PDF files

In [ ]:
%%sql -r dataframe_5
LIST @HEALTHCARE_AI_DEMO.RAW.S3_MEDICAL_DOCS;

---
## Step 3: Auto-Ingest with Snowpipe (5 min)

Snowpipe with `AUTO_INGEST = TRUE` creates a managed SQS queue. S3 event notifications trigger it automatically.

We only ingest **metadata** (file name, type, timestamp) — the actual PDFs stay in S3 and are read on-demand by AI functions.

In [ ]:
%%sql -r dataframe_6
USE DATABASE HEALTHCARE_AI_DEMO;

CREATE OR REPLACE TABLE RAW.FILES_LOG (
    FILE_ID       NUMBER AUTOINCREMENT PRIMARY KEY,
    FILE_NAME     VARCHAR NOT NULL,
    FILE_PATH     VARCHAR NOT NULL,
    FILE_TYPE     VARCHAR(10) NOT NULL,
    S3_EVENT_TIME TIMESTAMP_NTZ,
    LANDED_AT     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    IS_PROCESSED  BOOLEAN DEFAULT FALSE,
    PROCESSED_AT  TIMESTAMP_NTZ
);

CREATE OR REPLACE FILE FORMAT RAW.METADATA_ONLY_FORMAT
  TYPE = 'CSV' RECORD_DELIMITER = NONE FIELD_DELIMITER = NONE;

CREATE OR REPLACE PIPE RAW.PIPE_MEDICAL_DOCS AUTO_INGEST = TRUE AS
  COPY INTO RAW.FILES_LOG (FILE_NAME, FILE_PATH, FILE_TYPE, S3_EVENT_TIME)
  FROM (
    SELECT METADATA$FILENAME, METADATA$FILENAME, 'PDF', METADATA$START_SCAN_TIME
    FROM @RAW.S3_MEDICAL_DOCS
  )
  FILE_FORMAT = (FORMAT_NAME = 'RAW.METADATA_ONLY_FORMAT');

### Get SQS ARN for S3 Event Notifications

Copy the `notification_channel` value → configure S3 event notifications in AWS Console.

**AWS Console → S3 → Your bucket → Properties → Event notifications → Create:**
- Name: `snowpipe-pdfs`
- Prefix: `healthcare/pdfs/`
- Events: All object create events
- Destination: SQS queue → paste the ARN

In [ ]:
%%sql -r dataframe_7
SHOW PIPES IN SCHEMA RAW;

---
## Step 4: Ingest Files (2 min)

Refresh the pipe and use DIRECTORY() fallback to avoid waiting for async Snowpipe.

In [ ]:
%%sql -r dataframe_8
ALTER PIPE RAW.PIPE_MEDICAL_DOCS REFRESH;

-- Fallback: load directly from directory (Snowpipe is async, this is instant)
INSERT INTO RAW.FILES_LOG (FILE_NAME, FILE_PATH, FILE_TYPE, S3_EVENT_TIME)
  SELECT RELATIVE_PATH, RELATIVE_PATH, 'PDF', LAST_MODIFIED::TIMESTAMP_NTZ
  FROM DIRECTORY(@RAW.S3_MEDICAL_DOCS)
  WHERE RELATIVE_PATH != ''
    AND NOT EXISTS (SELECT 1 FROM RAW.FILES_LOG fl WHERE fl.FILE_NAME = RELATIVE_PATH);

In [ ]:
%%sql -r dataframe_9
SELECT FILE_TYPE, COUNT(*) AS FILES FROM RAW.FILES_LOG GROUP BY FILE_TYPE;

---
## Step 5: Explore Cortex AI Functions Interactively (15 min)

Before we build the automated pipeline, let's explore each AI function individually on a real PDF. This is the most powerful part of the demo — watch AI extract intelligence from a document in real-time.

### 5a. AI_PARSE_DOCUMENT — Extract text from a PDF

Reads the PDF directly from S3 via `TO_FILE()` and returns the text content.

In [ ]:
%%sql -r dataframe_10
SELECT
  FILE_NAME,
  AI_PARSE_DOCUMENT(
    TO_FILE('@RAW.S3_MEDICAL_DOCS', FILE_NAME),
    OBJECT_CONSTRUCT('mode', 'OCR')
  ):content::VARCHAR AS PARSED_TEXT
FROM RAW.FILES_LOG
WHERE FILE_TYPE = 'PDF'
LIMIT 1;

### 5b. AI_EXTRACT — Pull structured fields from unstructured text

Give it a schema of what you want, and it extracts structured data from free text. No training needed.

In [ ]:
%%sql -r dataframe_11
SELECT
  FILE_NAME,
  AI_EXTRACT(
    AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')):content::VARCHAR,
    OBJECT_CONSTRUCT(
      'patient_name', 'string: full name of the patient',
      'diagnosis', 'string: primary diagnosis',
      'medications', 'array: list of medications',
      'provider_name', 'string: doctor name'
    )
  ) AS EXTRACTED
FROM RAW.FILES_LOG
WHERE FILE_TYPE = 'PDF'
LIMIT 1;

### 5c. AI_CLASSIFY — Categorize documents automatically

Pass a list of categories and the AI assigns the best match. Zero-shot classification.

In [ ]:
%%sql -r dataframe_12
SELECT
  FILE_NAME,
  AI_CLASSIFY(
    AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')):content::VARCHAR,
    ARRAY_CONSTRUCT('Lab Report', 'Discharge Summary', 'Prescription', 'Radiology Report', 'Clinical Notes')
  ):labels[0]::VARCHAR AS CATEGORY
FROM RAW.FILES_LOG
WHERE FILE_TYPE = 'PDF'
LIMIT 3;

### 5d. AI_SUMMARIZE — Get a concise summary

In [ ]:
%%sql -r dataframe_13
SELECT
  FILE_NAME,
  SNOWFLAKE.CORTEX.SUMMARIZE(
    AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')):content::VARCHAR
  ) AS SUMMARY
FROM RAW.FILES_LOG
WHERE FILE_TYPE = 'PDF'
LIMIT 1;

### 5e. AI_REDACT — Remove PII automatically

Patient names, dates of birth, addresses — all redacted in one function call.

In [ ]:
%%sql -r dataframe_14
SELECT
  FILE_NAME,
  AI_REDACT(
    AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')):content::VARCHAR
  ) AS REDACTED_TEXT
FROM RAW.FILES_LOG
WHERE FILE_TYPE = 'PDF'
LIMIT 1;

---
## Step 6: Build the Full AI Pipeline (10 min)

Now we wrap all those functions into a stored procedure that processes every PDF in one go.

In [ ]:
%%sql -r dataframe_15
CREATE OR REPLACE TABLE PROCESSED.PDF_INTELLIGENCE (
    DOC_ID NUMBER AUTOINCREMENT PRIMARY KEY,
    FILE_ID NUMBER NOT NULL, FILE_NAME VARCHAR NOT NULL,
    PROCESSED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    PARSED_TEXT VARCHAR(16777216),
    EXTRACTED_FIELDS VARIANT,
    DOC_CATEGORY VARCHAR(100),
    SENTIMENT_SCORE FLOAT,
    SUMMARY VARCHAR(10000),
    REDACTED_TEXT VARCHAR(16777216),
    KEY_INSIGHTS VARCHAR(10000),
    EMBEDDING VECTOR(FLOAT, 768)
);

In [ ]:
%%sql -r dataframe_16
CREATE OR REPLACE PROCEDURE PROCESSED.PROCESS_PDF_FILES()
  RETURNS VARCHAR LANGUAGE SQL EXECUTE AS CALLER
AS $$
BEGIN
  INSERT INTO PROCESSED.PDF_INTELLIGENCE (
      FILE_ID, FILE_NAME, PARSED_TEXT, EXTRACTED_FIELDS, DOC_CATEGORY,
      SENTIMENT_SCORE, SUMMARY, REDACTED_TEXT, KEY_INSIGHTS, EMBEDDING)
  SELECT f.FILE_ID, f.FILE_NAME,
      AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')):content::VARCHAR,
      AI_EXTRACT(AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')):content::VARCHAR,
        OBJECT_CONSTRUCT('patient_name','string: full name','diagnosis','string: primary diagnosis','medications','array: medications','provider_name','string: doctor name','date_of_service','string: date')),
      AI_CLASSIFY(AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')):content::VARCHAR,
        ARRAY_CONSTRUCT('Lab Report','Discharge Summary','Prescription','Radiology Report','Clinical Notes','Pathology Report')):labels[0]::VARCHAR,
      SNOWFLAKE.CORTEX.SENTIMENT(AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')):content::VARCHAR),
      SNOWFLAKE.CORTEX.SUMMARIZE(AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')):content::VARCHAR),
      AI_REDACT(AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')):content::VARCHAR),
      AI_COMPLETE('mistral-large2', CONCAT('Provide 3 key clinical insights from this document:\n', AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')):content::VARCHAR)),
      AI_EMBED('snowflake-arctic-embed-m-v1.5', AI_PARSE_DOCUMENT(TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')):content::VARCHAR)
  FROM RAW.FILES_LOG f
  WHERE f.FILE_TYPE = 'PDF' AND f.IS_PROCESSED = FALSE;
  UPDATE RAW.FILES_LOG SET IS_PROCESSED = TRUE, PROCESSED_AT = CURRENT_TIMESTAMP() WHERE FILE_TYPE = 'PDF' AND IS_PROCESSED = FALSE;
  RETURN 'Done: ' || CURRENT_TIMESTAMP()::VARCHAR;
END; $$;

### Run it (takes ~5 minutes for 6 PDFs)

In [ ]:
%%sql -r dataframe_17
CALL PROCESSED.PROCESS_PDF_FILES();

### Explore the results

In [ ]:
%%sql -r dataframe_18
SELECT FILE_NAME, DOC_CATEGORY, SENTIMENT_SCORE,
  EXTRACTED_FIELDS:patient_name::VARCHAR AS PATIENT,
  EXTRACTED_FIELDS:diagnosis::VARCHAR AS DIAGNOSIS,
  LEFT(SUMMARY, 200) AS SUMMARY_PREVIEW
FROM PROCESSED.PDF_INTELLIGENCE;

---
## Step 7: Cortex Search — Semantic Retrieval (3 min)

Cortex Search indexes your AI-enriched text and lets you search by *meaning*, not just keywords. It powers the agent's document retrieval.

In [ ]:
%%sql -r dataframe_19
CREATE OR REPLACE CORTEX SEARCH SERVICE PROCESSED.PDF_SEARCH
  ON SEARCH_TEXT
  ATTRIBUTES DOC_CATEGORY, PATIENT_NAME, DIAGNOSIS
  WAREHOUSE = HEALTHCARE_AI_WH
  TARGET_LAG = '1 hour'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-m-v1.5'
AS (
    SELECT DOC_ID, FILE_NAME,
      CONCAT(COALESCE(PARSED_TEXT,''), '\nSUMMARY: ', COALESCE(SUMMARY,''), '\nINSIGHTS: ', COALESCE(KEY_INSIGHTS,'')) AS SEARCH_TEXT,
      DOC_CATEGORY,
      COALESCE(EXTRACTED_FIELDS:patient_name::VARCHAR, 'Unknown') AS PATIENT_NAME,
      COALESCE(EXTRACTED_FIELDS:diagnosis::VARCHAR, '') AS DIAGNOSIS
    FROM PROCESSED.PDF_INTELLIGENCE
);

---
## Step 8: Structured Data for Cortex Analyst (3 min)

Load sample structured healthcare data (providers, patients, claims). This is what Cortex Analyst will query via natural language.

In [ ]:
%%sql -r dataframe_20
USE SCHEMA ANALYTICS;

CREATE OR REPLACE TABLE ANALYTICS.PROVIDERS (
    PROVIDER_ID NUMBER PRIMARY KEY, PROVIDER_NAME VARCHAR NOT NULL, SPECIALTY VARCHAR(100) NOT NULL,
    FACILITY_NAME VARCHAR(200), CITY VARCHAR(100), STATE VARCHAR(50)
);
INSERT INTO ANALYTICS.PROVIDERS VALUES
  (1,'Dr. Sarah Chen','Internal Medicine','Mercy General Hospital','Hartford','CT'),
  (2,'Dr. James Okafor','Cardiology','Mercy General Hospital','Hartford','CT'),
  (3,'Dr. Maria Rodriguez','Orthopedics','St. Luke''s Medical Center','New Haven','CT'),
  (4,'Dr. Robert Kim','Neurology','Yale New Haven Hospital','New Haven','CT'),
  (5,'Dr. Aisha Patel','Oncology','Hartford Healthcare Cancer','Hartford','CT'),
  (6,'Dr. David Thompson','Psychiatry','Silver Hill Hospital','New Canaan','CT');

CREATE OR REPLACE TABLE ANALYTICS.PATIENTS (
    PATIENT_ID NUMBER PRIMARY KEY, FIRST_NAME VARCHAR(100) NOT NULL, LAST_NAME VARCHAR(100) NOT NULL,
    INSURANCE_PLAN VARCHAR(200), PRIMARY_PROVIDER_ID NUMBER
);
INSERT INTO ANALYTICS.PATIENTS VALUES
  (101,'John','Whitfield','Aetna PPO',1),(102,'Margaret','Sullivan','UnitedHealth',2),
  (103,'David','Park','Cigna',3),(104,'Sandra','Williams','Anthem Blue Cross',1),
  (105,'Roberto','Garcia','Medicare Advantage',2),(106,'Linda','Brown','Medicare Advantage',5);

CREATE OR REPLACE TABLE ANALYTICS.CLAIMS (
    CLAIM_ID NUMBER PRIMARY KEY, PATIENT_ID NUMBER NOT NULL, PROVIDER_ID NUMBER NOT NULL,
    SERVICE_DATE DATE NOT NULL, PROCEDURE_DESC VARCHAR(500), DIAGNOSIS_DESC VARCHAR(500),
    BILLED_AMOUNT NUMBER(10,2), PAID_AMOUNT NUMBER(10,2), CLAIM_STATUS VARCHAR(50)
);
INSERT INTO ANALYTICS.CLAIMS VALUES
  (1001,101,1,'2024-06-10','Office visit','Essential hypertension',185.00,113.60,'Approved'),
  (1002,101,2,'2024-06-15','Electrocardiogram','Essential hypertension',350.00,224.00,'Approved'),
  (1003,102,2,'2024-07-01','Echocardiography','Atherosclerotic heart disease',1200.00,768.00,'Approved'),
  (1004,103,3,'2024-07-10','Total knee replacement','Osteoarthritis right knee',45000.00,30400.00,'Approved'),
  (1005,105,2,'2024-08-05','Left heart catheterization','Atherosclerotic heart disease',8500.00,5760.00,'Approved'),
  (1006,106,5,'2024-08-12','Surgical pathology','Malignant neoplasm breast',425.00,272.00,'Approved'),
  (1007,104,1,'2024-09-10','Office visit','Acute respiratory infection',250.00,156.00,'Approved'),
  (1008,105,2,'2024-11-01','Coronary artery bypass','Atherosclerotic heart disease',85000.00,57600.00,'Approved'),
  (1009,101,1,'2024-11-10','Office visit','Essential hypertension',250.00,156.00,'Approved'),
  (1010,103,3,'2025-01-08','Post-op follow-up','Osteoarthritis right knee',185.00,113.60,'Denied');

---
## Step 9: Semantic View for Cortex Analyst (3 min)

The Semantic View tells Cortex Analyst how your tables relate, what each column means, and what metrics can be computed. This enables natural-language-to-SQL.

> **Alternative:** You can also create this via the Snowsight wizard at **AI & ML → Semantic Views → Create**.

In [ ]:
%%sql -r dataframe_21
CREATE OR REPLACE SEMANTIC VIEW ANALYTICS.HEALTHCARE_ANALYTICS
  TABLES (
    PATIENTS AS ANALYTICS.PATIENTS PRIMARY KEY (PATIENT_ID) COMMENT = 'Patient demographics',
    PROVIDERS AS ANALYTICS.PROVIDERS PRIMARY KEY (PROVIDER_ID) COMMENT = 'Doctor directory',
    CLAIMS AS ANALYTICS.CLAIMS PRIMARY KEY (CLAIM_ID) COMMENT = 'Medical claims and billing'
  )
  RELATIONSHIPS (
    claims_to_patients AS CLAIMS (PATIENT_ID) REFERENCES PATIENTS,
    claims_to_providers AS CLAIMS (PROVIDER_ID) REFERENCES PROVIDERS
  )
  DIMENSIONS (
    PATIENTS.first_name AS FIRST_NAME COMMENT = 'Patient first name',
    PATIENTS.last_name AS LAST_NAME COMMENT = 'Patient last name',
    PATIENTS.insurance_plan AS INSURANCE_PLAN COMMENT = 'Insurance plan (Aetna PPO, UnitedHealth, Cigna, Anthem Blue Cross, Medicare Advantage)',
    PROVIDERS.provider_name AS PROVIDER_NAME COMMENT = 'Doctor name',
    PROVIDERS.specialty AS SPECIALTY COMMENT = 'Medical specialty',
    CLAIMS.procedure_desc AS PROCEDURE_DESC COMMENT = 'Medical procedure performed',
    CLAIMS.diagnosis_desc AS DIAGNOSIS_DESC COMMENT = 'Diagnosis',
    CLAIMS.claim_status AS CLAIM_STATUS COMMENT = 'Status: Approved or Denied',
    CLAIMS.service_date AS SERVICE_DATE COMMENT = 'Date of service'
  )
  METRICS (
    CLAIMS.total_billed AS SUM(BILLED_AMOUNT) COMMENT = 'Total billed in dollars',
    CLAIMS.total_paid AS SUM(PAID_AMOUNT) COMMENT = 'Total paid by insurance in dollars',
    CLAIMS.claim_count AS COUNT(CLAIM_ID) COMMENT = 'Number of claims',
    CLAIMS.avg_billed AS AVG(BILLED_AMOUNT) COMMENT = 'Average billed amount'
  )
  COMMENT = 'Healthcare analytics for Cortex Analyst';

---
## Step 10: Create the Cortex Agent (3 min)

The agent combines **Cortex Analyst** (structured queries) with **Cortex Search** (document retrieval) into a single conversational interface. It decides which tool to use based on the question.

In [ ]:
%%sql -r dataframe_22
CREATE OR REPLACE AGENT ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT
  FROM SPECIFICATION
$$
models:
  orchestration: auto
instructions:
  system: >
    You are a healthcare intelligence assistant. You help users explore patient data,
    claims, and medical documents. Provide concise, clinically relevant answers.
  orchestration: >
    Use HealthcareAnalyst for quantitative questions about patients, providers, claims, and costs.
    Use PDFSearch for questions about medical document content, diagnoses, and clinical findings.
  sample_questions:
    - question: "Which providers have the highest total billed amounts?"
    - question: "Find documents mentioning diabetes or hypertension"
    - question: "What is the average claim amount by specialty?"
tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: HealthcareAnalyst
      description: "Queries structured healthcare data: patients, providers, claims, costs."
  - tool_spec:
      type: cortex_search
      name: PDFSearch
      description: "Searches AI-processed PDF medical documents for clinical content."
tool_resources:
  HealthcareAnalyst:
    semantic_view: "HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_ANALYTICS"
    execution_environment:
      type: warehouse
      warehouse: "HEALTHCARE_AI_WH"
  PDFSearch:
    name: "HEALTHCARE_AI_DEMO.PROCESSED.PDF_SEARCH"
$$;

---
## Step 11: Test the Agent (10 min)

### Option A: Snowflake Intelligence (recommended)
Go to **AI & ML → Intelligence → New Agent** → select `HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT`

### Option B: SQL

**Structured query** (routes to Cortex Analyst):

In [ ]:
%%sql -r dataframe_23
SELECT TRY_PARSE_JSON(
  SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
    'HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT',
    $${"messages": [{"role": "user", "content": [{"type": "text", "text": "Which providers have the highest total billed amounts?"}]}]}$$
  )
) AS RESPONSE;

**Document search** (routes to Cortex Search):

In [ ]:
%%sql -r dataframe_24
SELECT TRY_PARSE_JSON(
  SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
    'HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT',
    $${"messages": [{"role": "user", "content": [{"type": "text", "text": "Find documents mentioning hypertension"}]}]}$$
  )
) AS RESPONSE;

**Cross-tool query** (agent uses both):

In [ ]:
%%sql -r dataframe_25
SELECT TRY_PARSE_JSON(
  SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
    'HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT',
    $${"messages": [{"role": "user", "content": [{"type": "text", "text": "What diagnoses appear in both the claims data and the medical documents?"}]}]}$$
  )
) AS RESPONSE;

---
## Step 12: Deploy a Streamlit Chat App (5 min)

Deploy a real Streamlit app accessible in Snowsight. This creates a shareable chat interface powered by the agent that anyone in your org can use.

In [ ]:
CREATE OR REPLACE STAGE HEALTHCARE_AI_DEMO.ANALYTICS.STREAMLIT_APP
  DIRECTORY = (ENABLE = TRUE)
  ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

In [ ]:
from snowflake.snowpark.context import get_active_session
import io

session = get_active_session()

app_code = """import streamlit as st
from snowflake.snowpark.context import get_active_session
import json

session = get_active_session()

st.set_page_config(page_title="Healthcare Intelligence", page_icon="\U0001f3e5")
st.title("\U0001f3e5 Healthcare Intelligence Agent")
st.caption("Ask questions about patients, claims, or medical documents")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Ask anything..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    payload = json.dumps({"messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}]})

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            result = session.sql(f"""
                SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
                    'HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT',
                    $${payload}$$
                )
            """).collect()
            response = json.loads(result[0][0])
            text_parts = [item["text"] for item in response.get("content", []) if item.get("type") == "text"]
            answer = "\n".join(text_parts) if text_parts else "No response received."
            st.markdown(answer)
            st.session_state.messages.append({"role": "assistant", "content": answer})
"""

env_yml = "name: streamlit\nchannels:\n  - snowflake\ndependencies:\n  - streamlit>=1.31.0\n  - snowflake-snowpark-python>=1.11.0\n"

session.file.put_stream(io.BytesIO(app_code.encode("utf-8")), "@HEALTHCARE_AI_DEMO.ANALYTICS.STREAMLIT_APP/streamlit_app.py", overwrite=True, auto_compress=False)
session.file.put_stream(io.BytesIO(env_yml.encode("utf-8")), "@HEALTHCARE_AI_DEMO.ANALYTICS.STREAMLIT_APP/environment.yml", overwrite=True, auto_compress=False)
print("Streamlit app files uploaded to stage")

In [ ]:
CREATE OR REPLACE STREAMLIT HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_CHAT_APP
  ROOT_LOCATION = '@HEALTHCARE_AI_DEMO.ANALYTICS.STREAMLIT_APP'
  MAIN_FILE = 'streamlit_app.py'
  QUERY_WAREHOUSE = 'HEALTHCARE_AI_WH'
  TITLE = 'Healthcare Intelligence Chat';

### Your app is live!

Go to **Projects > Streamlit** in Snowsight and open **Healthcare Intelligence Chat**.

You now have:
- A **Cortex Agent** available in Snowflake Intelligence (AI & ML > Intelligence)
- A **Streamlit Chat App** shareable with anyone in your org (Projects > Streamlit)

---
## Bonus: Bring Your Own Document

Upload a new PDF to your S3 bucket, then run the cells below to process it and ask the agent about it.

In [ ]:
-- After uploading a new PDF to s3://<YOUR_BUCKET_NAME>/healthcare/pdfs/
ALTER PIPE RAW.PIPE_MEDICAL_DOCS REFRESH;

INSERT INTO RAW.FILES_LOG (FILE_NAME, FILE_PATH, FILE_TYPE, S3_EVENT_TIME)
  SELECT RELATIVE_PATH, RELATIVE_PATH, 'PDF', LAST_MODIFIED::TIMESTAMP_NTZ
  FROM DIRECTORY(@RAW.S3_MEDICAL_DOCS)
  WHERE RELATIVE_PATH != ''
    AND NOT EXISTS (SELECT 1 FROM RAW.FILES_LOG fl WHERE fl.FILE_NAME = RELATIVE_PATH AND fl.FILE_TYPE = 'PDF');

-- Strip any S3 prefix that Snowpipe may have added
UPDATE RAW.FILES_LOG
  SET FILE_NAME = REGEXP_REPLACE(FILE_NAME, '^healthcare/(pdfs|txt|audio)/', ''),
      FILE_PATH = REGEXP_REPLACE(FILE_PATH, '^healthcare/(pdfs|txt|audio)/', '')
  WHERE FILE_NAME LIKE 'healthcare/%';

CALL PROCESSED.PROCESS_PDF_FILES();

Now ask the agent about your new document!

---
## Cleanup

In [ ]:
%%sql -r dataframe_27
-- Uncomment to remove all lab objects:
-- DROP DATABASE IF EXISTS HEALTHCARE_AI_DEMO;
-- DROP WAREHOUSE IF EXISTS HEALTHCARE_AI_WH;
-- DROP STORAGE INTEGRATION IF EXISTS HEALTHCARE_S3_INTEGRATION;